# MECHA — TLA+ verification with the real TLC model checker

Rupp & Solland (2026). *MECHA: Formal Verification of Conjunctive Human-AI Execution Governance Using TLA+*

This notebook runs the **actual TLC model checker** (Java) on the MECHA spec `EFAVALO_Epistemic.tla` — not the Python BFS reproduction (`mecha_checker.py` / `P6_MECHA_Colab.ipynb`).

**What it does:**
1. Installs Java + downloads `tla2tools.jar` (official TLA+ tools)
2. Writes the v1.1 spec + TLC config
3. Runs TLC — checks `ConjunctiveIntegrity`, `SeparationOfDuties`, `NoDoubleFinalize`
4. Re-runs the **v1.0 bug variant** (missing `~vetoed[op]` guard) so TLC itself finds the double-finalize counterexample

Expected: **v1.1 → 0 violations**, **v1.0 → `NoDoubleFinalize` violated with a counterexample trace**.

Runtime: ~3–6 minutes on a free Colab CPU instance (3 operators, MaxTime=10).

## 1 · Setup — Java + TLA+ tools

In [ ]:
import shutil, subprocess, sys

# Colab ships with OpenJDK; install if missing
if shutil.which('java') is None:
    !apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless
!java -version

# Official TLA+ tools (TLC model checker)
!wget -q -N https://github.com/tlaplus/tlaplus/releases/latest/download/tla2tools.jar
!ls -la tla2tools.jar

## 2 · The MECHA spec (v1.1 — verified)

In [ ]:
%%writefile EFAVALO_Epistemic.tla
---- MODULE EFAVALO_Epistemic ----
(* MECHA: Conjunctive Human-AI Execution Governance Gate
   Verified invariants: ConjunctiveIntegrity, SeparationOfDuties, NoDoubleFinalize
   v1.1 — includes ~vetoed[op] guard in AllowAction (bug fixed from v1.0)
*)
EXTENDS Integers, FiniteSets

CONSTANTS Operators, MaxTime
ASSUME Operators \in SUBSET STRING /\ Operators # {}

VARIABLES
    machineState,    \* [op -> "Ready" | "Unstable"]
    evidenceState,   \* [op -> "Verified" | "Unverified"]
    human_state,     \* [op -> [acuity: 0..5, cogLoad: 0..5, affBias: 0..5]]
    authorityState,  \* [op -> "Authorized" | "Unauthorized"]
    action_executed, \* [op -> BOOLEAN]
    vetoed,          \* [op -> BOOLEAN]
    authorizers,     \* [op -> SUBSET Operators]
    sensorIntegrity, \* "Clean" | "Anomaly"
    time             \* 0..MaxTime

vars == <<machineState, evidenceState, human_state, authorityState,
          action_executed, vetoed, authorizers, sensorIntegrity, time>>

\* H condition: IsCapable checks all three HSRS components
IsCapable(op) ==
    LET acuity  == human_state[op].acuity
        cogLoad == human_state[op].cogLoad
        affBias == human_state[op].affBias
    IN  acuity  <= 3
     /\ cogLoad <= 2
     /\ affBias <= 2
     /\ (acuity + cogLoad + affBias) <= 7

\* Full conjunctive gate: M /\ E /\ H /\ A + Separation of Duties
MECHAGateOpen(op) ==
    /\ machineState[op]   = "Ready"
    /\ evidenceState[op]  = "Verified"
    /\ IsCapable(op)
    /\ authorityState[op] = "Authorized"
    /\ Cardinality(authorizers[op]) >= 1
    /\ \E auth \in authorizers[op] : auth # op

\* ── INIT ──────────────────────────────────────────────────────────────────
Init ==
    /\ machineState    = [op \in Operators |-> "Ready"]
    /\ evidenceState   = [op \in Operators |-> "Verified"]
    /\ human_state     = [op \in Operators |-> [acuity |-> 1, cogLoad |-> 1, affBias |-> 1]]
    /\ authorityState  = [op \in Operators |-> "Authorized"]
    /\ action_executed = [op \in Operators |-> FALSE]
    /\ vetoed          = [op \in Operators |-> FALSE]
    /\ authorizers     = [op \in Operators |-> {}]
    /\ sensorIntegrity = "Clean"
    /\ time            = 0

\* ── TRANSITIONS ───────────────────────────────────────────────────────────
Tick == time < MaxTime /\ time' = time + 1
     /\ UNCHANGED <<machineState, evidenceState, human_state, authorityState,
                    action_executed, vetoed, authorizers, sensorIntegrity>>

AddAuthorizer(op, auth) ==
    /\ auth \in Operators
    /\ auth # op
    /\ authorizers' = [authorizers EXCEPT ![op] = @ \union {auth}]
    /\ UNCHANGED <<machineState, evidenceState, human_state, authorityState,
                   action_executed, vetoed, sensorIntegrity, time>>

\* KEY GUARD: ~vetoed[op] prevents double-finalize (was missing in v1.0)
AllowAction(op) ==
    /\ ~action_executed[op]
    /\ ~vetoed[op]
    /\ MECHAGateOpen(op)
    /\ action_executed' = [action_executed EXCEPT ![op] = TRUE]
    /\ UNCHANGED <<machineState, evidenceState, human_state, authorityState,
                   vetoed, authorizers, sensorIntegrity, time>>

VetoAction(op) ==
    /\ ~action_executed[op]
    /\ ~vetoed[op]
    /\ vetoed' = [vetoed EXCEPT ![op] = TRUE]
    /\ UNCHANGED <<machineState, evidenceState, human_state, authorityState,
                   action_executed, authorizers, sensorIntegrity, time>>

\* Degradation/recovery only applies pre-execution. Once AllowAction has fired
\* for op, that operator's decision state is frozen.
DegradeMachine(op) ==
    /\ ~action_executed[op]
    /\ machineState[op] = "Ready"
    /\ machineState' = [machineState EXCEPT ![op] = "Unstable"]
    /\ UNCHANGED <<evidenceState, human_state, authorityState,
                   action_executed, vetoed, authorizers, sensorIntegrity, time>>

RecoverMachine(op) ==
    /\ ~action_executed[op]
    /\ machineState[op] = "Unstable"
    /\ machineState' = [machineState EXCEPT ![op] = "Ready"]
    /\ UNCHANGED <<evidenceState, human_state, authorityState,
                   action_executed, vetoed, authorizers, sensorIntegrity, time>>

InvalidateEvidence(op) ==
    /\ ~action_executed[op]
    /\ evidenceState[op] = "Verified"
    /\ evidenceState' = [evidenceState EXCEPT ![op] = "Unverified"]
    /\ UNCHANGED <<machineState, human_state, authorityState,
                   action_executed, vetoed, authorizers, sensorIntegrity, time>>

DegradeHuman(op) ==
    /\ ~action_executed[op]
    /\ IsCapable(op)
    /\ human_state' = [human_state EXCEPT ![op] =
            [acuity |-> 4, cogLoad |-> 3, affBias |-> 3]]
    /\ UNCHANGED <<machineState, evidenceState, authorityState,
                   action_executed, vetoed, authorizers, sensorIntegrity, time>>

RevokeAuthority(op) ==
    /\ ~action_executed[op]
    /\ authorityState[op] = "Authorized"
    /\ authorityState' = [authorityState EXCEPT ![op] = "Unauthorized"]
    /\ UNCHANGED <<machineState, evidenceState, human_state,
                   action_executed, vetoed, authorizers, sensorIntegrity, time>>

Next ==
    \/ Tick
    \/ \E op \in Operators : AllowAction(op)
    \/ \E op \in Operators : VetoAction(op)
    \/ \E op \in Operators, auth \in Operators : AddAuthorizer(op, auth)
    \/ \E op \in Operators : DegradeMachine(op)
    \/ \E op \in Operators : RecoverMachine(op)
    \/ \E op \in Operators : InvalidateEvidence(op)
    \/ \E op \in Operators : DegradeHuman(op)
    \/ \E op \in Operators : RevokeAuthority(op)

Spec == Init /\ [][Next]_vars

\* ── INVARIANTS ────────────────────────────────────────────────────────────
ConjunctiveIntegrity ==
    \A op \in Operators :
        action_executed[op] =>
            ( machineState[op]   = "Ready"
           /\ evidenceState[op]  = "Verified"
           /\ IsCapable(op)
           /\ authorityState[op] = "Authorized" )

SeparationOfDuties ==
    \A op \in Operators :
        action_executed[op] => \E auth \in authorizers[op] : auth # op

NoDoubleFinalize ==
    \A op \in Operators :
        ~(action_executed[op] /\ vetoed[op])

====


In [ ]:
%%writefile EFAVALO_Epistemic.cfg
CONSTANTS
    Operators = {"op1", "op2", "op3"}
    MaxTime = 10

SPECIFICATION Spec

INVARIANTS
    ConjunctiveIntegrity
    SeparationOfDuties
    NoDoubleFinalize


## 3 · Run TLC on v1.1

All three invariants should hold — exit code 0, no errors.

In [ ]:
result_v11 = subprocess.run(
    ['java', '-XX:+UseParallelGC', '-cp', 'tla2tools.jar', 'tlc2.TLC',
     '-workers', 'auto', '-deadlock',
     '-config', 'EFAVALO_Epistemic.cfg', 'EFAVALO_Epistemic.tla'],
    capture_output=True, text=True)
print(result_v11.stdout)
print(result_v11.stderr, file=sys.stderr)
print(f'TLC exit code: {result_v11.returncode}')

In [ ]:
import re

def summarize(run, label):
    out = run.stdout
    gen = re.search(r'(\d+) states generated', out)
    dis = re.search(r'(\d+) distinct states', out)
    violated = re.findall(r'Invariant (\w+) is violated', out)
    ok = run.returncode == 0 and not violated
    print(f'── {label} ' + '─' * (50 - len(label)))
    print(f'  states generated : {gen.group(1) if gen else "?"}')
    print(f'  distinct states  : {dis.group(1) if dis else "?"}')
    print(f'  violations       : {violated if violated else "none"}')
    print(f'  verdict          : {"✅ ALL INVARIANTS HOLD" if ok else "❌ VIOLATION FOUND"}')
    return {'label': label, 'generated': int(gen.group(1)) if gen else 0,
            'distinct': int(dis.group(1)) if dis else 0, 'violated': violated}

s11 = summarize(result_v11, 'MECHA v1.1 (correct)')

## 4 · The v1.0 bug — let TLC find it

v1.0 of `AllowAction` was missing the `~vetoed[op]` guard. TLC should now violate `NoDoubleFinalize` and print the exact counterexample trace: an operator is vetoed, then the action executes anyway.

In [ ]:
%%writefile EFAVALO_Epistemic_v10.tla
---- MODULE EFAVALO_Epistemic_v10 ----
(* MECHA: Conjunctive Human-AI Execution Governance Gate
   Verified invariants: ConjunctiveIntegrity, SeparationOfDuties, NoDoubleFinalize
   v1.1 — includes ~vetoed[op] guard in AllowAction (bug fixed from v1.0)
*)
EXTENDS Integers, FiniteSets

CONSTANTS Operators, MaxTime
ASSUME Operators \in SUBSET STRING /\ Operators # {}

VARIABLES
    machineState,    \* [op -> "Ready" | "Unstable"]
    evidenceState,   \* [op -> "Verified" | "Unverified"]
    human_state,     \* [op -> [acuity: 0..5, cogLoad: 0..5, affBias: 0..5]]
    authorityState,  \* [op -> "Authorized" | "Unauthorized"]
    action_executed, \* [op -> BOOLEAN]
    vetoed,          \* [op -> BOOLEAN]
    authorizers,     \* [op -> SUBSET Operators]
    sensorIntegrity, \* "Clean" | "Anomaly"
    time             \* 0..MaxTime

vars == <<machineState, evidenceState, human_state, authorityState,
          action_executed, vetoed, authorizers, sensorIntegrity, time>>

\* H condition: IsCapable checks all three HSRS components
IsCapable(op) ==
    LET acuity  == human_state[op].acuity
        cogLoad == human_state[op].cogLoad
        affBias == human_state[op].affBias
    IN  acuity  <= 3
     /\ cogLoad <= 2
     /\ affBias <= 2
     /\ (acuity + cogLoad + affBias) <= 7

\* Full conjunctive gate: M /\ E /\ H /\ A + Separation of Duties
MECHAGateOpen(op) ==
    /\ machineState[op]   = "Ready"
    /\ evidenceState[op]  = "Verified"
    /\ IsCapable(op)
    /\ authorityState[op] = "Authorized"
    /\ Cardinality(authorizers[op]) >= 1
    /\ \E auth \in authorizers[op] : auth # op

\* ── INIT ──────────────────────────────────────────────────────────────────
Init ==
    /\ machineState    = [op \in Operators |-> "Ready"]
    /\ evidenceState   = [op \in Operators |-> "Verified"]
    /\ human_state     = [op \in Operators |-> [acuity |-> 1, cogLoad |-> 1, affBias |-> 1]]
    /\ authorityState  = [op \in Operators |-> "Authorized"]
    /\ action_executed = [op \in Operators |-> FALSE]
    /\ vetoed          = [op \in Operators |-> FALSE]
    /\ authorizers     = [op \in Operators |-> {}]
    /\ sensorIntegrity = "Clean"
    /\ time            = 0

\* ── TRANSITIONS ───────────────────────────────────────────────────────────
Tick == time < MaxTime /\ time' = time + 1
     /\ UNCHANGED <<machineState, evidenceState, human_state, authorityState,
                    action_executed, vetoed, authorizers, sensorIntegrity>>

AddAuthorizer(op, auth) ==
    /\ auth \in Operators
    /\ auth # op
    /\ authorizers' = [authorizers EXCEPT ![op] = @ \union {auth}]
    /\ UNCHANGED <<machineState, evidenceState, human_state, authorityState,
                   action_executed, vetoed, sensorIntegrity, time>>

\* v1.0 BUG VARIANT: ~vetoed[op] guard removed — double-finalize possible
AllowAction(op) ==
    /\ ~action_executed[op]
    /\ MECHAGateOpen(op)
    /\ action_executed' = [action_executed EXCEPT ![op] = TRUE]
    /\ UNCHANGED <<machineState, evidenceState, human_state, authorityState,
                   vetoed, authorizers, sensorIntegrity, time>>

VetoAction(op) ==
    /\ ~action_executed[op]
    /\ ~vetoed[op]
    /\ vetoed' = [vetoed EXCEPT ![op] = TRUE]
    /\ UNCHANGED <<machineState, evidenceState, human_state, authorityState,
                   action_executed, authorizers, sensorIntegrity, time>>

\* Degradation/recovery only applies pre-execution. Once AllowAction has fired
\* for op, that operator's decision state is frozen.
DegradeMachine(op) ==
    /\ ~action_executed[op]
    /\ machineState[op] = "Ready"
    /\ machineState' = [machineState EXCEPT ![op] = "Unstable"]
    /\ UNCHANGED <<evidenceState, human_state, authorityState,
                   action_executed, vetoed, authorizers, sensorIntegrity, time>>

RecoverMachine(op) ==
    /\ ~action_executed[op]
    /\ machineState[op] = "Unstable"
    /\ machineState' = [machineState EXCEPT ![op] = "Ready"]
    /\ UNCHANGED <<evidenceState, human_state, authorityState,
                   action_executed, vetoed, authorizers, sensorIntegrity, time>>

InvalidateEvidence(op) ==
    /\ ~action_executed[op]
    /\ evidenceState[op] = "Verified"
    /\ evidenceState' = [evidenceState EXCEPT ![op] = "Unverified"]
    /\ UNCHANGED <<machineState, human_state, authorityState,
                   action_executed, vetoed, authorizers, sensorIntegrity, time>>

DegradeHuman(op) ==
    /\ ~action_executed[op]
    /\ IsCapable(op)
    /\ human_state' = [human_state EXCEPT ![op] =
            [acuity |-> 4, cogLoad |-> 3, affBias |-> 3]]
    /\ UNCHANGED <<machineState, evidenceState, authorityState,
                   action_executed, vetoed, authorizers, sensorIntegrity, time>>

RevokeAuthority(op) ==
    /\ ~action_executed[op]
    /\ authorityState[op] = "Authorized"
    /\ authorityState' = [authorityState EXCEPT ![op] = "Unauthorized"]
    /\ UNCHANGED <<machineState, evidenceState, human_state,
                   action_executed, vetoed, authorizers, sensorIntegrity, time>>

Next ==
    \/ Tick
    \/ \E op \in Operators : AllowAction(op)
    \/ \E op \in Operators : VetoAction(op)
    \/ \E op \in Operators, auth \in Operators : AddAuthorizer(op, auth)
    \/ \E op \in Operators : DegradeMachine(op)
    \/ \E op \in Operators : RecoverMachine(op)
    \/ \E op \in Operators : InvalidateEvidence(op)
    \/ \E op \in Operators : DegradeHuman(op)
    \/ \E op \in Operators : RevokeAuthority(op)

Spec == Init /\ [][Next]_vars

\* ── INVARIANTS ────────────────────────────────────────────────────────────
ConjunctiveIntegrity ==
    \A op \in Operators :
        action_executed[op] =>
            ( machineState[op]   = "Ready"
           /\ evidenceState[op]  = "Verified"
           /\ IsCapable(op)
           /\ authorityState[op] = "Authorized" )

SeparationOfDuties ==
    \A op \in Operators :
        action_executed[op] => \E auth \in authorizers[op] : auth # op

NoDoubleFinalize ==
    \A op \in Operators :
        ~(action_executed[op] /\ vetoed[op])

====


In [ ]:
%%writefile EFAVALO_Epistemic_v10.cfg
CONSTANTS
    Operators = {"op1", "op2", "op3"}
    MaxTime = 10

SPECIFICATION Spec

INVARIANTS
    ConjunctiveIntegrity
    SeparationOfDuties
    NoDoubleFinalize


In [ ]:
result_v10 = subprocess.run(
    ['java', '-XX:+UseParallelGC', '-cp', 'tla2tools.jar', 'tlc2.TLC',
     '-workers', 'auto', '-deadlock',
     '-config', 'EFAVALO_Epistemic_v10.cfg', 'EFAVALO_Epistemic_v10.tla'],
    capture_output=True, text=True)
print(result_v10.stdout)
print(f'TLC exit code: {result_v10.returncode}  (non-zero expected — violation)')

In [ ]:
s10 = summarize(result_v10, 'MECHA v1.0 (bug variant)')

## 5 · Comparison

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
labels = [s11['label'], s10['label']]

ax1.bar(labels, [s11['distinct'], s10['distinct']], color=['#22C55E', '#EF4444'])
ax1.set_title('Distinct states explored')
ax1.tick_params(axis='x', rotation=10)

ax2.bar(labels, [len(s11['violated']), len(s10['violated'])], color=['#22C55E', '#EF4444'])
ax2.set_title('Invariants violated')
ax2.set_yticks([0, 1])
ax2.tick_params(axis='x', rotation=10)

plt.suptitle('MECHA — real TLC verification: v1.1 vs v1.0')
plt.tight_layout()
plt.savefig('mecha_tlc_comparison.png', dpi=150)
plt.show()
print('Saved: mecha_tlc_comparison.png')

## Notes

- `Operators = {"op1", "op2"}`, `MaxTime = 3` (from the cfg) keeps the run small. Scale up by editing the cfg cell — e.g. 3 operators / `MaxTime = 10` for the full state space.
- `-deadlock` disables deadlock checking: terminal states (action executed or vetoed for all operators, clock exhausted) are expected and are not errors.
- The Python BFS reproduction lives in `mecha_checker.py`; the state counts differ from TLC because TLC canonicalizes states after symmetry/fingerprinting, while the Python checker enumerates the raw product. The **verdicts** (which invariants hold) must agree.
